# DeMoSearch · YOLO26m training
This notebook calls ordinary Python modules. Dataset preparation and training live in
`src/hold_detection/yolo_training.py`; no Python source is exported from this notebook.
The same functions run through `scripts/train_yolo26.py` on HPC.

Export a version of HoldClassificatore as **YOLO detection** from Roboflow, then
upload the ZIP to Drive. Select a GPU runtime for training.

In [ ]:
# @title Set up the environment (once)
import json, os, subprocess, sys, uuid
from pathlib import Path
from datetime import datetime, timezone

CODE_REF = 'main'  # @param {type: "string"}
WORK = Path('/content/demosearch-yolo26')
PROJECT = WORK / 'project'
WORK.mkdir(parents=True, exist_ok=True)
repository = 'https://github.com/DiasKzStrong/hold_detection.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', repository, str(PROJECT)], check=True)
if not (PROJECT / '.git').is_dir():
    raise RuntimeError('The project directory contains unrelated files; choose another WORK path.')
dirty = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
if dirty.strip():
    raise RuntimeError('Save local project changes before updating; they will not be overwritten.')
subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--quiet', 'origin', CODE_REF], check=True)
CODE_SHA = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
previous = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if previous != CODE_SHA and 'hold_detection' in sys.modules:
    raise RuntimeError('Source code has changed: restart the runtime to avoid mixing module versions.')
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--quiet', '--detach', CODE_SHA], check=True)
if str(PROJECT / 'src') not in sys.path: sys.path.insert(0, str(PROJECT / 'src'))
from hold_detection.notebook_runtime import prepare_runtime, run_cli, runtime_info
PYTHON = prepare_runtime(WORK, PROJECT)
MODEL_ENV = runtime_info(PYTHON, PROJECT)

def run_holds(*arguments, log_name='command'):
    return run_cli(PYTHON, PROJECT, list(arguments), WORK / 'logs' / (log_name + '.log'))

print('Ready ·', MODEL_ENV['gpu'] if MODEL_ENV['cuda_available'] else 'CPU baseline')
print('Kernel Python:', sys.version.split()[0], '| model Python:', MODEL_ENV['python'])
print('Code commit:', CODE_SHA, '| installation logs:', WORK / 'logs')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_SOURCE = Path('/content/drive/MyDrive/DeMoSearch/HoldClassificatore.zip')
PREPARED = WORK / 'datasets' / ('holds_' + uuid.uuid4().hex[:8])
run_holds('prepare-yolo', DATASET_SOURCE, '--output', PREPARED, log_name='prepare_yolo')
display(json.loads((PREPARED / 'dataset_report.json').read_text()))

In [ ]:
MODEL = 'yolo26m.pt'
EPOCHS = 3
IMAGE_SIZE = 1024
BATCH_SIZE = 8
RESUME = False
RUN = Path('/content/drive/MyDrive/DeMoSearch/yolo26m_smoke')

In [ ]:
args = ['train-yolo', '--data', PREPARED / 'data.yaml', '--output', RUN,
        '--model', MODEL, '--epochs', EPOCHS, '--imgsz', IMAGE_SIZE,
        '--batch', BATCH_SIZE, '--workers', 2, '--device', '0']
if RESUME: args += ['--resume']
run_holds(*args, log_name='train_yolo26')
print('Best weights:', RUN / 'weights/best.pt')

In [ ]:
run_holds('evaluate-yolo', '--data', PREPARED / 'data.yaml', '--output', RUN,
          '--split', 'val', '--device', '0', '--imgsz', IMAGE_SIZE,
          log_name='evaluate_yolo26')
display(json.loads((RUN / 'metrics_val.json').read_text()))

Start with 3 epochs, then use a **new run directory** and 100 epochs for a full
experiment. Resume applies to interrupted runs with the original epoch budget.
Checkpoints and logs are persistent on Drive. HoldClassificatore has six box-label
classes, without undercling/sidepull/opening direction. SAM3 supplies masks.

Full wall analysis (2D photo or 3D photos/video) uses `holds analyze-wall`.
See `docs/wall-pipeline.md` for the complete SLURM workflow and
`docs/yolo26-training.md` for training jobs.